# 6. Сургалтын үр дүнг сайн гаргах гиперпараметрийн тохиргоог хий

**Dataset:** MNIST  
**Model:** Simple MLP (784 → 128 → 64 → 10)  
**Туршлах гиперпараметр:** `learning_rate`, `batch_size`  
**Үнэлгээ:** Train/Val loss graph, Val accuracy

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

# Data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
full_train = datasets.MNIST("./data", train=True, download=True, transform=transform)
train_set, val_set = random_split(
    full_train, [50000, 10000],
    generator=torch.Generator().manual_seed(42),
)

# Train / evaluate
def train(model, loader, optimizer, loss_fn):
    model.train()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss_fn(model(x), y).backward()
        optimizer.step()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, n = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        n += y.size(0)
    return correct / n

# Train one config, return val accuracy
def train_and_validate(lr, batch_size, epochs=3):
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=256)

    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(28*28, 128), nn.ReLU(),
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, 10),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        train(model, train_loader, optimizer, loss_fn)
    return evaluate(model, val_loader)

# Grid search
lrs = [1e-2, 1e-3, 1e-4]
batch_sizes = [64, 128]

results = []
for lr in lrs:
    for bs in batch_sizes:
        acc = train_and_validate(lr, bs)
        print(f"lr={lr}, batch_size={bs} -> val_acc={acc:.4f}")
        results.append((acc, lr, bs))

best_acc, best_lr, best_bs = max(results)
print(f"\nBest: lr={best_lr}, batch_size={best_bs}, val_acc={best_acc:.4f}")